In [6]:
import pandas as pd

df = pd.read_csv('online_retail.csv', encoding='latin1')
print(df.shape)
print(df.dtypes)
df.head()

(541909, 8)
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2022-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2022-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [9]:
print("duplicates:",df.duplicated().sum())
print("Negative Quantity",(df['Quantity']<=0).sum())
print("Zero/Negative Price:",(df['UnitPrice']<=0).sum())
print("Cancelled invoices:",df['InvoiceNo'].str.startswith('C').sum())

duplicates: 5268
Negative Quantity 10624
Zero/Negative Price: 2517
Cancelled invoices: 9288


In [11]:
df=df.dropna(subset=['CustomerID'])
df=df[~df['InvoiceNo'].str.startswith('C')]
df=df[df['Quantity']>0]
df=df[df['UnitPrice']>0]
df=df.drop_duplicates()
print("Final shape:",df.shape)
print(df.isnull().sum())

Final shape: (392692, 8)
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [12]:
df['TotalPrice']=df['Quantity']*df['UnitPrice']
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['Month']=df['InvoiceDate'].dt.month
df['Year']=df["InvoiceDate"].dt.year    
print(df.head())


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  TotalPrice  \
0 2022-12-01 08:26:00       2.55     17850.0  United Kingdom       15.30   
1 2022-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34   
2 2022-12-01 08:26:00       2.75     17850.0  United Kingdom       22.00   
3 2022-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34   
4 2022-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34   

   Month  Year  
0     12  2022  
1     12  2022  
2     12  2022  
3     12  2022  
4     12  2022  


In [13]:
##top selling products
top_products=df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products)

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54319
JUMBO BAG RED RETROSPOT               46078
WHITE HANGING HEART T-LIGHT HOLDER    36706
ASSORTED COLOUR BIRD ORNAMENT         35263
PACK OF 72 RETROSPOT CAKE CASES       33670
POPCORN HOLDER                        30919
RABBIT NIGHT LIGHT                    27153
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64


In [14]:
##country wise sales
country_sales=df.groupby('Country')["TotalPrice"].sum().sort_values(ascending=False).head(10)
print(country_sales)

Country
United Kingdom    7285024.644
Netherlands        285446.340
EIRE               265262.460
Germany            228678.400
France             208934.310
Australia          138453.810
Spain               61558.560
Switzerland         56443.950
Belgium             41196.340
Sweden              38367.830
Name: TotalPrice, dtype: float64


In [15]:
monthly_sales=df.groupby(['Year','Month'])['TotalPrice'].sum().reset_index()
print(monthly_sales)


    Year  Month   TotalPrice
0   2022     12   570422.730
1   2023      1   568101.310
2   2023      2   446084.920
3   2023      3   594081.760
4   2023      4   468374.331
5   2023      5   677355.150
6   2023      6   660046.050
7   2023      7   598962.901
8   2023      8   644051.040
9   2023      9   950690.202
10  2023     10  1035642.450
11  2023     11  1156205.610
12  2023     12   517190.440


In [16]:
##customer purchase pattern
customer_pattern=df.groupby('CustomerID').agg(
    Total_Orders=('InvoiceNo','nunique'),
    Total_Spent=('TotalPrice','nunique'),
    Total_Items=('Quantity','sum')
).sort_values('Total_Spent',ascending=False).head(10)
print(customer_pattern)


            Total_Orders  Total_Spent  Total_Items
CustomerID                                        
12748.0              209          400        25287
14911.0              201          391        80240
14646.0               73          379       196915
14156.0               55          349        57768
17841.0              124          341        22834
14088.0               13          340        12665
14096.0               17          309        16352
15311.0               91          285        38147
14298.0               44          279        58343
18102.0               60          251        64124


In [18]:
##revenue analysis
total_revenue=df['TotalPrice'].sum()
avg_order_value=df.groupby('InvoiceNo')['TotalPrice'].sum().mean()
print(f"Total Revenue:£{total_revenue:,.2f}")
print(f"Average Order Value: £{avg_order_value:,.2f}")

Total Revenue:£8,887,208.89
Average Order Value: £479.56


In [20]:
##most active customers
active_customers=df.groupby('CustomerID')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)
print(active_customers)

CustomerID
12748.0    209
14911.0    201
17841.0    124
13089.0     97
14606.0     93
15311.0     91
12971.0     86
14646.0     73
16029.0     63
13408.0     62
Name: InvoiceNo, dtype: int64


In [24]:
import datetime as dt
reference_date=df['InvoiceDate'].max()+dt.timedelta(days=1)
rfm=df.groupby('CustomerID').agg(
    recency=('InvoiceDate',lambda x:(reference_date-x.max()).days),
    frequency=('InvoiceNo','nunique'),
    monetary=('TotalPrice','sum')
).reset_index()
print(rfm.head(10))
print(rfm.shape)


   CustomerID  recency  frequency  monetary
0     12346.0      326          1  77183.60
1     12347.0        2          7   4310.00
2     12348.0       75          4   1797.24
3     12349.0       19          1   1757.55
4     12350.0      310          1    334.40
5     12352.0       36          8   2506.04
6     12353.0      204          1     89.00
7     12354.0      232          1   1079.40
8     12355.0      214          1    459.40
9     12356.0       23          3   2811.43
(4338, 4)


In [27]:
##score each customer
rfm['R_Score']=pd.qcut(rfm['recency'],q=4,labels=[4,3,2,1])
rfm['F_Score']=pd.qcut(rfm['frequency'].rank(method='first'),q=4,labels=[1,2,3,4])
rfm['M_Score']=pd.qcut(rfm['monetary'],q=4,labels=[1,2,3,4])
rfm['RFM_Score']=rfm['R_Score'].astype(int)+rfm['F_Score'].astype(int)+rfm['M_Score'].astype(int)
print(rfm.head(10))


   CustomerID  recency  frequency  monetary R_Score F_Score M_Score  RFM_Score
0     12346.0      326          1  77183.60       1       1       4          6
1     12347.0        2          7   4310.00       4       4       4         12
2     12348.0       75          4   1797.24       2       3       4          9
3     12349.0       19          1   1757.55       3       1       4          8
4     12350.0      310          1    334.40       1       1       2          4
5     12352.0       36          8   2506.04       3       4       4         11
6     12353.0      204          1     89.00       1       1       1          3
7     12354.0      232          1   1079.40       1       1       3          5
8     12355.0      214          1    459.40       1       1       2          4
9     12356.0       23          3   2811.43       3       3       4         10


In [37]:
##segment customers
def segmentcustomer(score):
    if score >= 10:
        return 'High Value'
    elif score >= 7:
        return 'Medium Value'
    elif score >= 4:
        return 'Low Value'
    else:
        return 'At Risk'

rfm['Segment'] = rfm['RFM_Score'].apply(segmentcustomer)
print(rfm['Segment'].value_counts())
    

Segment
Low Value       1495
Medium Value    1276
High Value      1267
At Risk          300
Name: count, dtype: int64


In [38]:
##recommendation for a Specific Customer
def recommend_for_customer(customer_id,top_n=5):
    already_bought=df[df['CustomerID']==customer_id]['Description'].unique()
    segment=rfm[rfm['CustomerID']==customer_id]['Segment'].values[0]
    segment_customers=rfm[rfm['Segment']==segment]['CustomerID']
    segment_purchases=df[df['CustomerID'].isin(segment_customers)]
    top_products=(segment_purchases.groupby('Description')['Quantity'].sum().sort_values(ascending=False))
    recommendations=top_products[~top_products.index.isin(already_bought)].head(top_n)
    print(f"customer:{customer_id}")
    print(f"segment:{segment}")
    print(f"\nRecommended Products:")
    for i,(product,qty)in enumerate(recommendations.items(),1):
        print(f"{i}.{product}")
recommend_for_customer(17850.0)                                     


customer:17850.0
segment:Medium Value

Recommended Products:
1.WORLD WAR 2 GLIDERS ASSTD DESIGNS
2.FAIRY CAKE FLANNEL ASSORTED COLOUR
3.JUMBO BAG RED RETROSPOT
4.ASSORTED COLOUR BIRD ORNAMENT
5.ESSENTIAL BALM 3.5g TIN IN ENVELOPE


In [ ]:
##product recommendation
def recommend_products(segment_name,top_n=5):
    segment_customers=rfm[rfm['Segment']==segment_name]['CustomerID']
    segment_purchases=df[df['CustomerID'].isin(segment_customers)]
    top=segment_purchases.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(top_n)
    return top
print("==high value customers buy==")
print(recommend_products('High Value'))
print("\n===At risk customers buy===")
print(recommend_products('AtRisk'))


    ]

==high value customers buy==
Description
PAPER CRAFT , LITTLE BIRDIE          80995
WORLD WAR 2 GLIDERS ASSTD DESIGNS    40874
JUMBO BAG RED RETROSPOT              38230
POPCORN HOLDER                       28622
ASSORTED COLOUR BIRD ORNAMENT        27285
Name: Quantity, dtype: int64

===At risk customers buy===
Series([], Name: Quantity, dtype: int64)
